# Width-Scaling SGD Experiments with Causal Attention, d16 Excluded

Same task, optimizer, and width sweep as `colab_width_scaling_sgd_exclude_d8_d16.ipynb`,
with `use_attention: True` selecting the transformer architecture instead of the plain
residual MLP.

The transformer treats the problem as next-position prediction. The sequence is the 16
input bits followed by the target parities in binary-tree order (all degree-2 parities,
then degree-4, then degree-8). The last input position predicts the first degree-2
parity, the next position predicts the second, and so on. Every position — input bits
and intermediate answers alike — owns a learnable embedding vector that the value at
that position scales. Each of the `L` blocks is causal self-attention with a residual
connection followed by the same MLP block the residual net uses (also residual, with
`use_post_activation_linear` optional). There is no layer normalization. The unembedding
is a single position-independent `N -> 1` map.

Training is teacher-forced on the true parities; evaluation is autoregressive from the
input bits alone, so `metrics.csv` carries both `test_mse` (autoregressive) and
`test_mse_teacher_forced`.


## Setup

Mount Google Drive, clone the public repo, install it in editable mode, and define the run/plot directories.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/labofdoubt/feature-learning-parity-task.git"
REPO_DIR = Path("/content/feature-learning-parity-task")

DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_width_scaling_sgd_exclude_d16_attention")

RUNS_DIR = DRIVE_ROOT / "runs"
PLOTS_DIR = DRIVE_ROOT / "plots"
ANALYSIS_DIR = DRIVE_ROOT / "analysis"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!pip install -e .

## Train Width Sweep

One config per width, identical to the residual-MLP sweep except for `use_attention: True`
and `num_heads`. Existing final checkpoints are skipped unless `FORCE_RETRAIN = True`.


In [ ]:
from pathlib import Path

import yaml

from parity_net.config import load_config
from parity_net.train import train

WIDTHS = [2048]
FORCE_RETRAIN = True


def make_config(N: int, output_dir: Path) -> dict:
    return {
        "model": {
            "input_dim": 16,
            "relevant_dim": 8,
            "N": N,
            "L": 3,
            "activation": "half-tanh",
            "use_readout_barrier": False,
            "embedding_weight_variance": 1.0 / 32,
            "freeze_embedding": False,
            "hidden_weight_variance": 1.0 / N,
            "readout_weight_variance": 1.0 / N,
            "use_layerwise_readouts": False,
            "use_post_activation_linear": True,
            "bias": False,
            "use_attention": True,
            "num_heads": 1,
            "attention_logit_scale": "1/sqrt(d)",
            "autoregressive_feedback": "raw",
        },
        "task": {
            "input_dim": 16,
            "relevant_dim": 8,
            # "exclude_targets": ["d4", "d8", "d16"],
            "exclude_targets": ["d16"]
        },
        "training": {
            "num_steps": 50_000,
            "test_samples": 100_000,
            "batch_size": 512,
            "seed": 0,
            "device": "cuda",
            "dtype": "float32",
            "log_every": 1_000,
            "checkpoint_every": 10_000,
            "output_dir": str(output_dir),
            "barrier_c": None,
            "barrier_lambda": 10.0,
            "optimizer": {
                "name": "sgd",
                "lr": 1e-3,
                "lr_embedding": None,
                "lr_hidden": None,
                "lr_readout": None,
                "weight_decay": 1e-3,
                "wd_embedding": None,
                "wd_hidden": None,
                "wd_readout": None,
                "momentum": 0.9,
                "betas": [0.9, 0.999],
            },
        },
    }

def make_config_mup(N: int, output_dir: Path) -> dict:
    config = make_config(N, output_dir)
    optimizer = config["training"]["optimizer"]
    base_lr = optimizer["lr"]
    base_wd = optimizer["weight_decay"]
    config["model"]["readout_weight_variance"] = 1 / N**2
    # muP keeps query-key logits Theta(1) as head_dim grows with width.
    config["model"]["attention_logit_scale"] = "1/d"

    optimizer["lr_embedding"] = base_lr * N / 256
    optimizer["lr_hidden"] = base_lr
    optimizer["lr_readout"] = base_lr * 256 / N

    optimizer["wd_embedding"] = base_wd * 256 / N
    optimizer["wd_hidden"] = base_wd
    optimizer["wd_readout"] = base_wd * N / 256
    return config


CONFIG_FACTORY = make_config_mup  # Change to make_config for the unscaled sweep.

In [ ]:
config_paths = {}
for N in WIDTHS:
    run_dir = RUNS_DIR / f"N_{N}"
    run_dir.mkdir(parents=True, exist_ok=True)
    config_path = run_dir / "config.yaml"
    final_checkpoint = run_dir / "checkpoints" / "final.pt"

    if final_checkpoint.exists() and not FORCE_RETRAIN:
        config_paths[N] = config_path
        print(
            f"N={N}: final checkpoint exists, skipping training: {final_checkpoint}. "
            "Set FORCE_RETRAIN=True or choose a new DRIVE_ROOT to train with changed config values."
        )
        continue

    config = CONFIG_FACTORY(N, run_dir)
    with config_path.open("w") as f:
        yaml.safe_dump(config, f, sort_keys=False)
    config_paths[N] = config_path

    print(f"N={N}: training with {config_path}")
    final_path = train(load_config(config_path))
    print(f"N={N}: saved final checkpoint to {final_path}")

## Train/Test Curves

Read `metrics.csv` from each run, save one plot per width, and save combined train/test
plots across widths. Autoregressive test MSE is the headline metric; the teacher-forced
columns are plotted alongside so single-step error and compounded generation error stay
distinguishable.


In [ ]:
RUNS_DIR = DRIVE_ROOT / "runs"
PLOTS_DIR = DRIVE_ROOT / "plots"
ANALYSIS_DIR = DRIVE_ROOT / "analysis"

WIDTHS = [2048]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

USE_LOG_MSE_AXIS = True
USE_LOG_STEP_AXIS = True
DEGREE_SUFFIXES = ["", "_d2", "_d4", "_d8", "_d16"]
TEST_MSE_COLUMNS = [f"test_mse{suffix}" for suffix in DEGREE_SUFFIXES]
TEACHER_FORCED_COLUMNS = [f"test_mse_teacher_forced{suffix}" for suffix in DEGREE_SUFFIXES]


def axis_has_positive_data(ax, axis):
    for line in ax.lines:
        values = line.get_xdata() if axis == "x" else line.get_ydata()
        if len(values) and pd.Series(values).dropna().gt(0).any():
            return True
    return False


def maybe_log_y(ax):
    if USE_LOG_MSE_AXIS and axis_has_positive_data(ax, "y"):
        ax.set_yscale("log")


def maybe_log_x(ax):
    if USE_LOG_STEP_AXIS and axis_has_positive_data(ax, "x"):
        ax.set_xscale("log")


def plot_columns(df, columns, title, filename):
    present = [column for column in columns if column in df.columns]
    if not present:
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    for column in present:
        ax.plot(df["step"], df[column], label=column)
    ax.set_xlabel("Step")
    ax.set_ylabel("MSE")
    ax.set_title(title)
    maybe_log_x(ax)
    maybe_log_y(ax)
    ax.grid(True, alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150)
    plt.show()


metrics_by_width = {}
for N in WIDTHS:
    metrics_path = RUNS_DIR / f"N_{N}" / "metrics.csv"
    if not metrics_path.exists():
        print(f"N={N}: missing {metrics_path}")
        continue
    df = pd.read_csv(metrics_path)
    metrics_by_width[N] = df

    plot_columns(
        df,
        TEST_MSE_COLUMNS,
        f"Autoregressive test MSE by degree, N={N}",
        f"test_mse_by_degree_N_{N}.png",
    )
    plot_columns(
        df,
        TEACHER_FORCED_COLUMNS,
        f"Teacher-forced test MSE by degree, N={N}",
        f"test_mse_teacher_forced_by_degree_N_{N}.png",
    )
    plot_columns(
        df,
        ["train_mse", "test_mse", "test_mse_teacher_forced"],
        f"Train/Test MSE, N={N}",
        f"train_test_mse_N_{N}.png",
    )


def plot_across_widths(column, title, filename):
    if not any(column in df.columns for df in metrics_by_width.values()):
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    for N, df in metrics_by_width.items():
        if column in df.columns:
            ax.plot(df["step"], df[column], label=f"N={N}")
    ax.set_xlabel("Step")
    ax.set_ylabel(column)
    ax.set_title(title)
    maybe_log_x(ax)
    maybe_log_y(ax)
    ax.grid(True, alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150)
    plt.show()


plot_across_widths("test_mse", "Autoregressive Test MSE vs Step", "test_mse_by_width.png")
plot_across_widths(
    "test_mse_teacher_forced",
    "Teacher-Forced Test MSE vs Step",
    "test_mse_teacher_forced_by_width.png",
)
plot_across_widths("train_mse", "Train MSE vs Step", "train_mse_by_width.png")